In [1]:
import pandas as pd

In [2]:
sales_df = pd.read_parquet('../data/processed/05_lag_features.parquet')

In [3]:
sales_df.isnull().sum()

id                        0
item_id                   0
dept_id                   0
cat_id                    0
store_id                  0
state_id                  0
day                       0
sales                     0
date                      0
wm_yr_wk                  0
weekday                   0
wday                      0
month                     0
year                      0
d                         0
event_name_1       53631910
event_type_1       53631910
event_name_2       58205410
event_type_2       58205410
snap_CA                   0
snap_TX                   0
snap_WI                   0
sell_price         12299413
is_weekend                0
is_month_start            0
is_month_end              0
lag_1                 30490
lag_7                213430
lag_28               853720
rolling_mean_7        30490
rolling_mean_28       30490
dtype: int64

for event name and type column we will fill with "no event" as its categorical column. for lag and rolling features dropping the first 28 rows of (itemid,storeid) pair is preffered since its a small fraction of our dataset

In [4]:
cat_cols = ['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']

sales_df[cat_cols].fillna('No_events', inplace=True)

C:\Users\twofi\AppData\Local\Temp\ipykernel_24344\3996874531.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  sales_df[cat_cols].fillna('No_events', inplace=True)


,event_name_1,event_type_1,event_name_2,event_type_2
0,No_events,No_events,No_events,No_events
1,No_events,No_events,No_events,No_events
2,No_events,No_events,No_events,No_events
3,No_events,No_events,No_events,No_events
4,No_events,No_events,No_events,No_events
...,...,...,...,...
58327365,No_events,No_events,No_events,No_events
58327366,No_events,No_events,No_events,No_events
58327367,No_events,No_events,No_events,No_events
58327368,No_events,No_events,No_events,No_events


In [5]:
sales_df_before = len(sales_df)

In [26]:
sales_df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 58327370 entries, 0 to 58327369
Data columns (total 31 columns):
 #   Column           Dtype         
---  ------           -----         
 0   id               str           
 1   item_id          str           
 2   dept_id          str           
 3   cat_id           str           
 4   store_id         str           
 5   state_id         str           
 6   day              str           
 7   sales            uint16        
 8   date             datetime64[us]
 9   wm_yr_wk         uint16        
 10  weekday          str           
 11  wday             uint16        
 12  month            uint16        
 13  year             uint16        
 14  d                str           
 15  event_name_1     str           
 16  event_type_1     str           
 17  event_name_2     str           
 18  event_type_2     str           
 19  snap_CA          uint16        
 20  snap_TX          uint16        
 21  snap_WI          uint16        
 22  sel

In [6]:
cat_cols = sales_df.select_dtypes(include='str').columns.tolist()

In [7]:
sales_df[cat_cols] = sales_df[cat_cols].astype('category')

In [9]:
sales_df['sell_price'] = sales_df['sell_price'].astype('float32')

In [11]:
sales_df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 58327370 entries, 0 to 58327369
Data columns (total 31 columns):
 #   Column           Dtype         
---  ------           -----         
 0   id               category      
 1   item_id          category      
 2   dept_id          category      
 3   cat_id           category      
 4   store_id         category      
 5   state_id         category      
 6   day              category      
 7   sales            uint16        
 8   date             datetime64[us]
 9   wm_yr_wk         uint16        
 10  weekday          category      
 11  wday             uint16        
 12  month            uint16        
 13  year             uint16        
 14  d                category      
 15  event_name_1     category      
 16  event_type_1     category      
 17  event_name_2     category      
 18  event_type_2     category      
 19  snap_CA          uint16        
 20  snap_TX          uint16        
 21  snap_WI          uint16        
 22  sel

In [12]:
sales_df = sales_df.groupby(
    ['item_id', 'store_id'],
    group_keys=False
).nth(slice(28, None)).reset_index()

In [13]:
sales_df_after = len(sales_df)

In [15]:
print(f"Number of rows before: {sales_df_before}, Number of rows after: {sales_df_after}")
print(f"Number of rows dropped: {sales_df_before - sales_df_after}")

Number of rows before: 58327370, Number of rows after: 57473650
Number of rows dropped: 853720
